In [1]:
import requests
from bs4 import BeautifulSoup as bs

In [2]:
# 크롤링을 하기 위해 특정 주소에 요청을 보내서 html 문서를 받아온다
# 동기식 웹페이지에서 사용이 가능
# 비동기식 웹페이지에서는 selenium을 통해서 html 문서를 받아온다
res = requests.get('https://books.toscrape.com')

In [4]:
# 문자에서 특정한 부분을 추출하기 어렵기 때문에 BeautifulSoup 라이브러리를 이용해서 데이터를 추출한다
# 그러기 위해서 데이터 타입은 Beautiful의 타입으로 파싱
# python 코드 직역 -> BeautifulSoup class 객체를 생성해서 html text를 대입
soup = bs(res.text, 'html.parser')

# BeautifulSoup의 속성과 메서드
- 속성
    - 태그명 : html 문서 안에서 해당 태그의 첫번째 정보를 출력
- 메서드
    - find()
        - html 문서 내에서 특정 태그의 첫번째 정보를 출력
        - 태그, 속성을 기준으로 검색이 가능
        - 되돌려주는 데이터의 타입은 Tag Data
    - find_all()
        - html 문서 내에서 특정 태그의 모든 정보를 출력
        - 되돌려주는 데이터의 타입은 ResultSet
            - list의 형태와 흡사

In [6]:
# 해당 사이트에서 책의 이름과 가격을 크롤링
# 책에 대한 정보들은 태그의 이름은 ol, 속성은 class='row'인 태그 안에 존재
len(soup.find_all('ol', attrs={'class' : 'row'}))

1

In [8]:
# 태그의 이름이 ol이고 class가 row인 태그는 1개가 존재
# find() 함수를 이용하여 정보를 추출
ol_data = soup.find('ol', attrs={'class' : 'row'})

In [9]:
ol_data

<ol class="row">
<li class="col-xs-6 col-sm-4 col-md-3 col-lg-3">
<article class="product_pod">
<div class="image_container">
<a href="catalogue/a-light-in-the-attic_1000/index.html"><img alt="A Light in the Attic" class="thumbnail" src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg"/></a>
</div>
<p class="star-rating Three">
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
<i class="icon-star"></i>
</p>
<h3><a href="catalogue/a-light-in-the-attic_1000/index.html" title="A Light in the Attic">A Light in the ...</a></h3>
<div class="product_price">
<p class="price_color">Â£51.77</p>
<p class="instock availability">
<i class="icon-ok"></i>
    
        In stock
    
</p>
<form>
<button class="btn btn-primary btn-block" data-loading-text="Adding..." type="submit">Add to basket</button>
</form>
</div>
</article>
</li>
<li class="col-xs-6 col-sm-4 col-md-3 col-lg-3">
<article class="product_pod">
<div class="image_container">
<

In [10]:
# ol 태그는 리스트 태그이고 리스트 안에 각각의 항목을 li태그를 이용하여 지정
# ol_data 안에서 li 태그들을 모두 찾는다
len(ol_data.find_all('li'))

20

In [11]:
li_list = ol_data.find_all('li')

In [ ]:
# li_list에서 첫번째 데이터만을 이용해서 책의 이름과 가격을 출력
# 책의 이름은 h3 태그 안에 텍스트로 존재
# 특정 태그에서 텍스트만 추출하는 함수 -> 가져오다(get) + 문자(text)
li_list[0].find('h3').get_text()

'A Light in the ...'

In [24]:
# 책의 가격은 p 태그의 class = 'price_color'안에 텍스트로 존재
li_list[0].find('p', attrs={'class':'price_color'}).get_text()

'Â£51.77'

- 위의 코드를 활용해서 li_list를 반복하여 20개의 책의 제목과 가격을 데이터프레임을 생성

In [28]:
import pandas as pd

In [40]:
title = []
price = []
for i in range(len(li_list)):
    title.append(li_list[i].find('h3').get_text())
    price.append(li_list[i].find('p', attrs={'class':'price_color'}).get_text())
df = pd.DataFrame({
    'Title' : title,
    'Price' : price
})
df

,Title,Price
0,A Light in the ...,Â£51.77
1,Tipping the Velvet,Â£53.74
2,Soumission,Â£50.10
3,Sharp Objects,Â£47.82
4,Sapiens: A Brief History ...,Â£54.23
5,The Requiem Red,Â£22.65
6,The Dirty Little Secrets ...,Â£33.34
7,The Coming Woman: A ...,Â£17.93
8,The Boys in the ...,Â£22.60
9,The Black Maria,Â£52.15


In [42]:
df['Price'] = df['Price'].map(
    lambda x : x.lstrip('Â')
)
df

,Title,Price
0,A Light in the ...,£51.77
1,Tipping the Velvet,£53.74
2,Soumission,£50.10
3,Sharp Objects,£47.82
4,Sapiens: A Brief History ...,£54.23
5,The Requiem Red,£22.65
6,The Dirty Little Secrets ...,£33.34
7,The Coming Woman: A ...,£17.93
8,The Boys in the ...,£22.60
9,The Black Maria,£52.15


In [48]:
# li_list를 이용해서 반복문을 생성
book_list = []
for i in range(len(li_list)):
    # i : index 값
    # 책 제목 추출
    title = li_list[i].find('h3').get_text()
    # 책 가격 추출
    price = li_list[i].find('p', attrs={'p', 'price_color'}).get_text()
    book_list.append(
        {
            'Title' : title,
            'Price' : price
        }
    )

pd.DataFrame(book_list)

,Title,Price
0,A Light in the ...,Â£51.77
1,Tipping the Velvet,Â£53.74
2,Soumission,Â£50.10
3,Sharp Objects,Â£47.82
4,Sapiens: A Brief History ...,Â£54.23
5,The Requiem Red,Â£22.65
6,The Dirty Little Secrets ...,Â£33.34
7,The Coming Woman: A ...,Â£17.93
8,The Boys in the ...,Â£22.60
9,The Black Maria,Â£52.15


In [45]:
book_list2 = []
for li_data in li_list:
    # li_data -> li_list 각각의 원소가 대입
    title = li_data.find('h3').get_text()
    price = li_data.find('p', attrs={'class' : 'price_color'}).get_text()
    book_list2.append(
        {
            'Title' : title,
            'Price' : price
        }
    )
pd.DataFrame(book_list2)

,Title,Price
0,A Light in the ...,Â£51.77
1,Tipping the Velvet,Â£53.74
2,Soumission,Â£50.10
3,Sharp Objects,Â£47.82
4,Sapiens: A Brief History ...,Â£54.23
5,The Requiem Red,Â£22.65
6,The Dirty Little Secrets ...,Â£33.34
7,The Coming Woman: A ...,Â£17.93
8,The Boys in the ...,Â£22.60
9,The Black Maria,Â£52.15
